# General-capability analysis

Run `python scripts/run.py general-capabilities` first. This notebook reads its saved results and creates comparison tables; it does not load models or run evaluation.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd


# ============================================================
# Paths and display order
# ============================================================

RESULT_DIR = Path("data/general_capability_side_effects")
INPUT_PATH = RESULT_DIR / "general_capability_comparison.csv"
OUTPUT_DIR = RESULT_DIR / "paper_tables"

MODEL_ORDER = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
    "meta-llama/Llama-3.1-8B",
]

MODEL_DISPLAY: Dict[str, str] = {
    "google/gemma-2-2b": "Gemma-2-2B",
    "google/gemma-2-9b": "Gemma-2-9B",
    "google/gemma-3-4b-pt": "Gemma-3-4B-PT",
    "google/gemma-3-4b-it": "Gemma-3-4B-IT",
    "google/gemma-3-12b-pt": "Gemma-3-12B-PT",
    "google/gemma-3-12b-it": "Gemma-3-12B-IT",
    "meta-llama/Llama-3.1-8B": "Llama-3.1-8B",
}

BENCHMARK_ORDER = [
    "mmlu",
    "arc_challenge",
    "boolq",
    "piqa",
]

BENCHMARK_DISPLAY: Dict[str, str] = {
    "mmlu": "MMLU",
    "arc_challenge": "ARC-C",
    "boolq": "BoolQ",
    "piqa": "PIQA",
}


# ============================================================
# Loading and validation
# ============================================================

def load_results() -> pd.DataFrame:
    if not INPUT_PATH.exists():
        raise FileNotFoundError(
            f"Could not find {INPUT_PATH}. Run "
            "evaluate_general_capability_side_effects.py first."
        )

    df = pd.read_csv(INPUT_PATH)

    required = {
        "model",
        "benchmark",
        "vanilla_accuracy",
        "steered_accuracy",
        "delta_accuracy",
        "prediction_flip_rate",
        "correct_to_incorrect_rate",
        "incorrect_to_correct_rate",
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {INPUT_PATH}: {sorted(missing)}")

    df = df[
        df["model"].isin(MODEL_ORDER)
        & df["benchmark"].isin(BENCHMARK_ORDER)
    ].copy()

    duplicates = df.duplicated(["model", "benchmark"], keep=False)
    if duplicates.any():
        bad = df.loc[duplicates, ["model", "benchmark"]]
        raise ValueError(
            "Duplicate model-benchmark rows found:\n"
            f"{bad.to_string(index=False)}"
        )

    expected = {
        (model, benchmark)
        for model in MODEL_ORDER
        for benchmark in BENCHMARK_ORDER
    }
    observed = set(
        df[["model", "benchmark"]].itertuples(index=False, name=None)
    )
    missing_pairs = sorted(expected - observed)
    if missing_pairs:
        raise ValueError(
            "Some model-benchmark evaluations are missing. "
            f"Examples: {missing_pairs[:20]}"
        )

    return df


# ============================================================
# Main paper table
# ============================================================

def build_main_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Two rows per model:
      Vanilla | MMLU | ARC-C | BoolQ | PIQA | Average
      Steered | MMLU | ARC-C | BoolQ | PIQA | Average

    No delta row is included.
    """
    rows: List[Dict] = []

    for model in MODEL_ORDER:
        model_df = df[df["model"].eq(model)].set_index("benchmark")

        for setting, value_col in [
            ("Vanilla", "vanilla_accuracy"),
            ("Steered", "steered_accuracy"),
        ]:
            row = {
                "Model": MODEL_DISPLAY[model],
                "Setting": setting,
            }

            values = []
            for benchmark in BENCHMARK_ORDER:
                value = float(model_df.loc[benchmark, value_col])
                row[BENCHMARK_DISPLAY[benchmark]] = value
                values.append(value)

            row["Average"] = float(np.mean(values))
            rows.append(row)

    # Add all-model averages at the bottom.
    for setting, value_col in [
        ("Vanilla", "vanilla_accuracy"),
        ("Steered", "steered_accuracy"),
    ]:
        row = {
            "Model": "Average",
            "Setting": setting,
        }

        benchmark_means = []
        for benchmark in BENCHMARK_ORDER:
            value = float(
                df.loc[df["benchmark"].eq(benchmark), value_col].mean()
            )
            row[BENCHMARK_DISPLAY[benchmark]] = value
            benchmark_means.append(value)

        row["Average"] = float(np.mean(benchmark_means))
        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# Direct side-effect summary
# ============================================================

def build_side_effect_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    One row per model, summarizing capability change and prediction transitions
    across the four benchmarks.
    """
    summary = (
        df.groupby("model", as_index=False)
        .agg(
            Vanilla_Avg=("vanilla_accuracy", "mean"),
            Steered_Avg=("steered_accuracy", "mean"),
            Delta_Avg=("delta_accuracy", "mean"),
            Mean_Flip_Rate=("prediction_flip_rate", "mean"),
            Correct_to_Incorrect=("correct_to_incorrect_rate", "mean"),
            Incorrect_to_Correct=("incorrect_to_correct_rate", "mean"),
        )
    )

    summary["Model"] = summary["model"].map(MODEL_DISPLAY)
    summary["model_order"] = summary["model"].map(
        {model: i for i, model in enumerate(MODEL_ORDER)}
    )
    summary = summary.sort_values("model_order")

    summary = summary[
        [
            "Model",
            "Vanilla_Avg",
            "Steered_Avg",
            "Delta_Avg",
            "Mean_Flip_Rate",
            "Correct_to_Incorrect",
            "Incorrect_to_Correct",
        ]
    ].reset_index(drop=True)

    overall = {
        "Model": "Average",
        "Vanilla_Avg": float(summary["Vanilla_Avg"].mean()),
        "Steered_Avg": float(summary["Steered_Avg"].mean()),
        "Delta_Avg": float(summary["Delta_Avg"].mean()),
        "Mean_Flip_Rate": float(summary["Mean_Flip_Rate"].mean()),
        "Correct_to_Incorrect": float(
            summary["Correct_to_Incorrect"].mean()
        ),
        "Incorrect_to_Correct": float(
            summary["Incorrect_to_Correct"].mean()
        ),
    }

    return pd.concat(
        [summary, pd.DataFrame([overall])],
        ignore_index=True,
    )


def build_benchmark_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    One row per benchmark, averaged across models.
    """
    summary = (
        df.groupby("benchmark", as_index=False)
        .agg(
            Vanilla_Avg=("vanilla_accuracy", "mean"),
            Steered_Avg=("steered_accuracy", "mean"),
            Delta_Avg=("delta_accuracy", "mean"),
            Mean_Flip_Rate=("prediction_flip_rate", "mean"),
            Correct_to_Incorrect=("correct_to_incorrect_rate", "mean"),
            Incorrect_to_Correct=("incorrect_to_correct_rate", "mean"),
        )
    )

    summary["Benchmark"] = summary["benchmark"].map(BENCHMARK_DISPLAY)
    summary["benchmark_order"] = summary["benchmark"].map(
        {benchmark: i for i, benchmark in enumerate(BENCHMARK_ORDER)}
    )
    summary = summary.sort_values("benchmark_order")

    return summary[
        [
            "Benchmark",
            "Vanilla_Avg",
            "Steered_Avg",
            "Delta_Avg",
            "Mean_Flip_Rate",
            "Correct_to_Incorrect",
            "Incorrect_to_Correct",
        ]
    ].reset_index(drop=True)


# ============================================================
# LaTeX formatting
# ============================================================

def bold_better_scores(main_table: pd.DataFrame) -> pd.DataFrame:
    """
    Return a string-formatted copy where the better Vanilla/Steered score
    for each model and benchmark is bolded. Ties are bolded in both rows.
    """
    formatted = main_table.copy()

    numeric_cols = [
        BENCHMARK_DISPLAY[b] for b in BENCHMARK_ORDER
    ] + ["Average"]

    for col in numeric_cols:
        formatted[col] = formatted[col].map(lambda x: f"{float(x):.3f}")

    for model in formatted["Model"].unique():
        indices = formatted.index[formatted["Model"].eq(model)].tolist()
        if len(indices) != 2:
            continue

        for col in numeric_cols:
            values = [
                float(main_table.loc[idx, col])
                for idx in indices
            ]
            best = max(values)

            for idx, value in zip(indices, values):
                text = f"{value:.3f}"
                formatted.loc[idx, col] = (
                    rf"\textbf{{{text}}}"
                    if np.isclose(value, best)
                    else text
                )

    return formatted


def save_latex_table(
    df: pd.DataFrame,
    path: Path,
    float_format: bool = True,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    if float_format:
        latex = df.to_latex(
            index=False,
            escape=False,
            float_format=lambda x: f"{x:.3f}",
        )
    else:
        latex = df.to_latex(
            index=False,
            escape=False,
        )

    path.write_text(latex, encoding="utf-8")


# ============================================================
# Main
# ============================================================

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = load_results()

    main_table = build_main_table(df)
    side_effect_summary = build_side_effect_summary(df)
    benchmark_summary = build_benchmark_summary(df)
    latex_main_table = bold_better_scores(main_table)

    main_table.to_csv(
        OUTPUT_DIR / "general_capability_main_table.csv",
        index=False,
    )
    side_effect_summary.to_csv(
        OUTPUT_DIR / "general_capability_side_effect_summary.csv",
        index=False,
    )
    benchmark_summary.to_csv(
        OUTPUT_DIR / "general_capability_benchmark_summary.csv",
        index=False,
    )

    save_latex_table(
        latex_main_table,
        OUTPUT_DIR / "general_capability_main_table.tex",
        float_format=False,
    )
    save_latex_table(
        side_effect_summary,
        OUTPUT_DIR / "general_capability_side_effect_summary.tex",
        float_format=True,
    )
    save_latex_table(
        benchmark_summary,
        OUTPUT_DIR / "general_capability_benchmark_summary.tex",
        float_format=True,
    )

    print("\nMain general-capability table")
    print("=" * 120)
    print(
        main_table.to_string(
            index=False,
            float_format=lambda x: f"{x:.3f}",
        )
    )

    print("\nDirect side-effect summary")
    print("=" * 120)
    print(
        side_effect_summary.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}",
        )
    )

    print(f"\nSaved tables to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()